In [1]:
import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [2]:
# Patch sqlite3 with bundled modern version — required on Linux where system sqlite3 < 3.35.0.
# Same patch used in tests/conftest.py. Must run before any chromadb import.
if sys.platform == "linux":
    __import__("pysqlite3")
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

## 1. Cargar documentos desde Chroma

BM25Retriever necesita los documentos en memoria. Los cargamos directamente
desde la colección existente en ChromaDB usando el cliente nativo.

In [3]:
import chromadb

from researchos.domain.models import Document
from researchos.paths import CHROMA_DIR

COLLECTION_NAME = "papers"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(COLLECTION_NAME)

raw = collection.get(include=["documents", "metadatas"])

documents = [
    Document(doc_id=doc_id, text=text, metadata=metadata)
    for doc_id, text, metadata in zip(raw["ids"], raw["documents"], raw["metadatas"])
]

print(f"Documentos cargados: {len(documents)}")

Failed to reload module 'sqlite3' from file '/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/pysqlite3/__init__.py'
Traceback (most recent call last):
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/.pyenv/versions/3.11.8/lib/python3.11/importlib/__init__.py", line 148, in reload
    raise ImportError(msg.format(name), name=name)
ImportError: module pysqlite3 not in sys.modules
[autoreload of sqlite3 failed: Traceback (most recent call last):
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/exte

Documentos cargados: 754


## 2. Crear BM25Retriever y buscar

El índice se construye en el constructor — tokenización básica por defecto.

In [4]:
from researchos.infrastructure.retrieval.bm25 import BM25Retriever

bm25 = BM25Retriever(documents=documents)

query = "What is the primary function of the harness in an externalized agent architecture?"

results = await bm25.search(query=query, k=5)

for r in results:
    print(f"score: {r.score:.4f} | paper: {r.metadata.get('paper_id', '?')}")
    print(f"text:  {r.text[:200]}")
    print()

score: 15.5751 | paper: sam_musker_2024
text:  composition, but be aided by an 
increased availability of information in this condition. What appears 
as undiminished model performance in compositional conditions may 
be the result of a balanced n

score: 14.9431 | paper: sam_musker_2024
text:  example, one subject 
attains a below-average match to reference of 50% in the Distracted
condition despite being able to state that the task involves ‘‘Looking 
at other comparable entries to figure 

score: 14.6667 | paper: sam_musker_2024
text:  & Rathkopf, 2025).
The question of whether we require human-likeness of a mecha-
nism to declare human-level ‘‘competence’’ is ultimately not empiri-
cal, but rather demands philosophical consensus am

score: 14.5783 | paper: sam_musker_2024
text:   
appears significantly smaller (see sections 4.2 and 4.3 of Grattafiori et al. 
(2024)). That said, Llama 405B’s training being mostly text prediction does 
not guarantee that this is what underlies 

scor

## 3. Comparar BM25 vs búsqueda vectorial

Misma query, mismo k — observa qué documentos recupera cada estrategia.

In [5]:
from researchos.infrastructure.retrieval.chroma import ChromaVectorStore
from researchos.infrastructure.retrieval.embedder import LocalEmbedder

embedder = LocalEmbedder()
chroma = ChromaVectorStore(embedder=embedder, collection_name=COLLECTION_NAME)

vector_results = await chroma.search(query=query, k=5)

print("=== Vector search ===")
for r in vector_results:
    print(f"score: {r.score:.4f} | paper: {r.metadata.get('paper_id', '?')}")
    print(f"text:  {r.text[:200]}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Vector search ===
score: 0.7266 | paper: hehai_lin_2025
text:  elf-improvement constitute a pivotal research tra-
jectory for enhancing agent capabilities through an
intra-agent feedback loop (Renze and Guven, 2024;
Shinn et al., 2023; Dou et al., 2024; Zhang et 

score: 0.7248 | paper: hehai_lin_2025
text:  -agent learn-
ing. For example, MALT (Motwani et al., 2024)
designs a sequential multi-agent system (MAS)
consisting of Generator, Verifier, and Refiner,
each independently trained to sample trajector

score: 0.7122 | paper: hehai_lin_2025
text:   to investigate whether and how interaction
within a multi-agent environment can enhance the
independent problem-solving capabilities of each
individual LLM. In this context, all agents share
the same

score: 0.7106 | paper: imad_aouali_2026
text:  mpting and agent protocols
This appendix documents the prompts used to generate benchmark queries and to run recommendation
agents. All prompts are fixed across models and experimental condi

In [6]:
bm25_ids  = {r.doc_id for r in results}
vector_ids = {r.doc_id for r in vector_results}

print(f"Solo en BM25:    {bm25_ids - vector_ids}")
print(f"Solo en vector:  {vector_ids - bm25_ids}")
print(f"En ambos:        {bm25_ids & vector_ids}")

Solo en BM25:    {'sam_musker_2024_166', 'sam_musker_2024_127', 'sam_musker_2024_219', 'hehai_lin_2025_187', 'sam_musker_2024_148'}
Solo en vector:  {'hehai_lin_2025_142', 'hehai_lin_2025_70', 'hehai_lin_2025_8', 'imad_aouali_2026_159', 'hehai_lin_2025_150'}
En ambos:        set()
